# 01 — Czyszczenie danych

## Cel

W tym notebooku wykonuję pierwszy etap pracy z datasetem Lending Club.

Na tym etapie:
- wczytuję dane,
- sprawdzam ich strukturę,
- analizuję braki,
- sprawdzam duplikaty,
- wykonuję tylko proste i oczywiste czyszczenie.

Bardziej złożone operacje, takie jak imputacja, kodowanie kategorii i skalowanie,
zostawiam do notebooka `02_transformacja.ipynb`.


## 1. Import bibliotek

In [1]:
# Path służy do tworzenia i obsługi ścieżek do plików i folderów.
# Dzięki temu łatwiej wskazać, skąd wczytujemy dane i gdzie zapisujemy wyniki.
from pathlib import Path

import pandas as pd


## 2. Wczytanie danych

Ścieżkę zapisuję w osobnej zmiennej, żeby później łatwo można było ją zmienić.
`TARGET` przechowuje nazwę kolumny, którą docelowo chcę przewidywać.


In [2]:
# Ścieżka do surowego datasetu.
# Plik znajduje się w katalogu data należącym do naszego projektu.
DATA_PATH = Path("data/loans_full_schema.csv")

# Ścieżka do pliku, w którym zapiszemy dataset po wykonaniu czyszczenia.
OUTPUT_PATH = Path("data/02_po_czyszczeniu.csv")

# Nazwa kolumny, którą później będziemy przewidywać za pomocą modelu.
TARGET = "interest_rate"

df = pd.read_csv(DATA_PATH)

# Tworzę kopię danych surowych.
# Dzięki temu w razie potrzeby mogę porównać dane przed i po czyszczeniu.
df_raw = df.copy()


## 3. Pierwsze spojrzenie na dataset

In [3]:
# Pierwsze 5 wierszy pozwala szybko zobaczyć, jak wyglądają dane.
df.head()


,emp_title,emp_length,state,homeownership,annual_income,verified_income,debt_to_income,annual_income_joint,verification_income_joint,debt_to_income_joint,...,sub_grade,issue_month,loan_status,initial_listing_status,disbursement_method,balance,paid_total,paid_principal,paid_interest,paid_late_fees
0,global config engineer,3.0,NJ,MORTGAGE,90000.0,Verified,18.01,NaN,NaN,NaN,...,C3,Mar-2018,Current,whole,Cash,27015.86,1999.33,984.14,1015.19,0.0
1,warehouse office clerk,10.0,HI,RENT,40000.0,Not Verified,5.04,NaN,NaN,NaN,...,C1,Feb-2018,Current,whole,Cash,4651.37,499.12,348.63,150.49,0.0
2,assembly,3.0,WI,RENT,40000.0,Source Verified,21.15,NaN,NaN,NaN,...,D1,Feb-2018,Current,fractional,Cash,1824.63,281.80,175.37,106.43,0.0
3,customer service,1.0,PA,RENT,30000.0,Not Verified,10.16,NaN,NaN,NaN,...,A3,Jan-2018,Current,whole,Cash,18853.26,3312.89,2746.74,566.15,0.0
4,security supervisor,10.0,CA,RENT,35000.0,Verified,57.96,57000.0,Verified,37.66,...,C3,Mar-2018,Current,whole,Cash,21430.15,2324.65,1569.85,754.80,0.0


In [4]:
# shape zwraca liczbę wierszy i kolumn.
print("Kształt datasetu:", df.shape)
print("Liczba wierszy:", df.shape[0])
print("Liczba kolumn:", df.shape[1])


Kształt datasetu: (10000, 55)
Liczba wierszy: 10000
Liczba kolumn: 55


In [5]:
# info() pokazuje m.in. nazwy kolumn, typy danych
# oraz liczbę wartości niepustych.
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 55 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   emp_title                         9167 non-null   str    
 1   emp_length                        9183 non-null   float64
 2   state                             10000 non-null  str    
 3   homeownership                     10000 non-null  str    
 4   annual_income                     10000 non-null  float64
 5   verified_income                   10000 non-null  str    
 6   debt_to_income                    9976 non-null   float64
 7   annual_income_joint               1495 non-null   float64
 8   verification_income_joint         1455 non-null   str    
 9   debt_to_income_joint              1495 non-null   float64
 10  delinq_2y                         10000 non-null  int64  
 11  months_since_last_delinq          4342 non-null   float64
 12  earliest_credit_

### Wnioski po `df.info()`

Dataset zawiera 10 000 wierszy i 55 kolumn.

Występują trzy główne typy danych:
- `float64` — 17 kolumn,
- `int64` — 25 kolumn,
- `str` — 13 kolumn.

Większość kolumn nie zawiera brakujących wartości, ale w kilku kolumnach widoczne są braki, np.:
`emp_title`, `emp_length`, `debt_to_income`, `annual_income_joint`,
`verification_income_joint`, `debt_to_income_joint`,
`months_since_last_delinq`, `months_since_90d_late`,
`months_since_last_credit_inquiry` oraz `num_accounts_120d_past_due`.

Kolumna targetowa `interest_rate` ma 10 000 wartości niepustych, więc na tym etapie nie ma potrzeby usuwać wierszy z powodu brakującego targetu.

Zużycie pamięci wynosi około 4.2 MB, więc dataset jest niewielki i można wygodnie analizować go w całości.

In [6]:
# describe() pokazuje podstawowe statystyki kolumn numerycznych.
df.describe().T


,count,mean,std,min,25%,50%,75%,max
emp_length,9183.0,5.930306,3.703734,0.00,2.0000,6.000,10.0000,1.000000e+01
annual_income,10000.0,79222.148412,64734.290492,0.00,45000.0000,65000.000,95000.0000,2.300000e+06
debt_to_income,9976.0,19.308192,15.004851,0.00,11.0575,17.570,25.0025,4.690900e+02
annual_income_joint,1495.0,127914.571244,70168.375404,19200.00,86833.5000,113000.000,151545.5000,1.100000e+06
debt_to_income_joint,1495.0,19.979304,8.054781,0.32,14.1600,19.720,25.5000,3.998000e+01
delinq_2y,10000.0,0.216000,0.683660,0.00,0.0000,0.000,0.0000,1.300000e+01
months_since_last_delinq,4342.0,36.760709,21.634939,1.00,19.0000,34.000,53.0000,1.180000e+02
earliest_credit_line,10000.0,2001.290000,7.795510,1963.00,1997.0000,2003.000,2006.0000,2.015000e+03
inquiries_last_12m,10000.0,1.958200,2.380130,0.00,0.0000,1.000,3.0000,2.900000e+01
total_credit_lines,10000.0,22.679600,11.885439,2.00,14.0000,21.000,29.0000,8.700000e+01


### Wnioski po `describe()`

Funkcja `describe()` pokazuje podstawowe statystyki dla kolumn numerycznych:
liczbę wartości (`count`), średnią (`mean`), odchylenie standardowe (`std`),
wartość minimalną i maksymalną oraz kwartyle 25%, 50% i 75%.

Dla targetu `interest_rate`:
- średnie oprocentowanie wynosi około 12.43%,
- mediana wynosi 11.98%,
- minimalne oprocentowanie to 5.31%,
- maksymalne oprocentowanie to 30.94%.

W kilku kolumnach widać bardzo duże wartości maksymalne w porównaniu z medianą
i kwartylami, np. `annual_income`, `debt_to_income` czy
`total_collection_amount_ever`. Mogą to być wartości odstające, ale ich dokładną
analizę zostawiam do etapu EDA.

W kolumnie `num_accounts_120d_past_due` wszystkie dostępne wartości są równe 0,
więc warto później sprawdzić, czy ta kolumna wnosi jakąkolwiek informację do modelu.

## 4. Sprawdzenie targetu

Targetem jest `interest_rate`.

Najpierw sprawdzam:
- czy taka kolumna istnieje,
- jaki ma typ,
- czy posiada brakujące wartości.


In [7]:
print("Czy target istnieje:", TARGET in df.columns)
print("Typ targetu:", df[TARGET].dtype)
print("Braki w targecie:", df[TARGET].isna().sum())

df[TARGET].describe()


Czy target istnieje: True
Typ targetu: float64
Braki w targecie: 0


count    10000.000000
mean        12.427524
std          5.001105
min          5.310000
25%          9.430000
50%         11.980000
75%         15.050000
max         30.940000
Name: interest_rate, dtype: float64

### Wnioski po sprawdzeniu targetu

Kolumna `interest_rate` istnieje i ma typ numeryczny `float64`,
więc nadaje się jako target do regresji liniowej.

Nie zawiera brakujących wartości, dlatego nie trzeba usuwać żadnych wierszy
z powodu braku targetu.

Oprocentowanie mieści się w zakresie od 5.31% do 30.94%,
a jego mediana wynosi 11.98%.

## 5. Brakujące wartości

Najpierw liczę braki. Nie uzupełniam ich automatycznie.

W tym etapie chcę przede wszystkim wiedzieć:
- które kolumny mają braki,
- ile ich jest,
- jaki procent kolumny stanowią.

Trudniejsze decyzje dotyczące uzupełniania braków zostawiam do transformacji.


In [8]:
# isna() sprawdza, które wartości są brakujące (NaN).
# sum() zlicza liczbę braków osobno w każdej kolumnie.
braki = df.isna().sum()

# len(df) zwraca liczbę wierszy w DataFrame.
# Dzielę liczbę braków przez liczbę wszystkich wierszy,
# mnożę przez 100 i otrzymuję procent braków w każdej kolumnie.
# round(2) zaokrągla wynik do dwóch miejsc po przecinku.
procent_brakow = (braki / len(df) * 100).round(2)

# Tworzę nowy DataFrame, w którym zestawiam
# liczbę braków oraz ich procent dla każdej kolumny.
tabela_brakow = pd.DataFrame({
    "braki": braki,
    "procent_brakow": procent_brakow
})

# Wybieram tylko kolumny, które mają co najmniej jeden brak,
# a następnie sortuję je od największego procentu braków do najmniejszego.
tabela_brakow[tabela_brakow["braki"] > 0].sort_values(
    "procent_brakow",
    ascending=False
)


,braki,procent_brakow
verification_income_joint,8545,85.45
annual_income_joint,8505,85.05
debt_to_income_joint,8505,85.05
months_since_90d_late,7715,77.15
months_since_last_delinq,5658,56.58
months_since_last_credit_inquiry,1271,12.71
emp_title,833,8.33
emp_length,817,8.17
num_accounts_120d_past_due,318,3.18
debt_to_income,24,0.24


### Wnioski z analizy brakujących wartości

Brakujące wartości występują w 10 kolumnach.

Najwięcej braków znajduje się w kolumnach związanych ze wspólnym wnioskiem:
`verification_income_joint`, `annual_income_joint` oraz `debt_to_income_joint`.
Braki przekraczają tam 85% obserwacji.

Może to wynikać z tego, że większość pożyczek była składana jako wnioski indywidualne,
więc dane dotyczące wspólnego dochodu nie miały dla nich zastosowania.
Sprawdzę to później przed podjęciem decyzji o uzupełnianiu lub usuwaniu tych wartości.

Dużo braków występuje również w kolumnach:
`months_since_90d_late` oraz `months_since_last_delinq`.
W tym przypadku brak może oznaczać, że dane zdarzenie wcześniej nie wystąpiło,
dlatego również nie należy automatycznie uzupełniać tych wartości.

Najmniej braków ma `debt_to_income` — tylko 24 wartości, czyli 0.24% danych.

Na tym etapie nie uzupełniam jeszcze braków. Najpierw sprawdzam ich znaczenie,
a decyzje dotyczące imputacji pozostawiam do etapu transformacji.

### Braki w targecie

W modelu nadzorowanym wiersz bez wartości targetu nie może służyć do trenowania modelu.
Jeżeli takie wiersze występują, usuwam tylko je.


In [9]:
liczba_przed = len(df)

df = df.dropna(subset=[TARGET])

liczba_po = len(df)

print("Liczba wierszy przed:", liczba_przed)
print("Liczba wierszy po:", liczba_po)
print("Usunięto:", liczba_przed - liczba_po)


Liczba wierszy przed: 10000
Liczba wierszy po: 10000
Usunięto: 0


## 6. Duplikaty

Sprawdzam dokładne duplikaty całych wierszy.

Nie usuwam rekordów tylko dlatego, że jakaś pojedyncza kolumna ma taką samą wartość.
Dwie różne osoby mogą przecież mieć np. ten sam dochód albo ten sam cel pożyczki.


In [10]:
liczba_duplikatow = df.duplicated().sum()

print("Liczba dokładnych duplikatów:", liczba_duplikatow)


Liczba dokładnych duplikatów: 0


In [11]:
# Jeżeli dokładne duplikaty istnieją, usuwam ich kolejne wystąpienia.
# Jeżeli wynik wynosi 0, ta operacja niczego nie zmieni.
df = df.drop_duplicates()

print("Kształt po sprawdzeniu duplikatów:", df.shape)


Kształt po sprawdzeniu duplikatów: (10000, 55)


## 7. Proste czyszczenie tekstu

W kolumnie `emp_title` mogą występować zbędne spacje na początku lub końcu tekstu.

`str.strip()` usuwa tylko takie zewnętrzne spacje.
Nie zmienia tekstu znajdującego się wewnątrz nazwy stanowiska.


In [12]:
# Najpierw sprawdzam, czy kolumna "emp_title" istnieje w DataFrame (employment title).
# Dzięki temu kod nie zgłosi błędu, gdyby kolumny nie było.
if "emp_title" in df.columns:

    # str.strip() usuwa zbędne spacje z początku i końca tekstu.
    # Nie usuwa spacji znajdujących się wewnątrz nazwy stanowiska.
    # .str Specjalny „dostęp do metod tekstowych” dla całej kolumny
    df["emp_title"] = df["emp_title"].str.strip()

## 8. Typy danych

Sprawdzam typy po podstawowym czyszczeniu.

Na tym etapie nie zmieniam typów na siłę.
Jeżeli kolumna jest już poprawnie rozpoznana jako liczba lub tekst,
pozostawiam ją bez zmian.

Ewentualne zmiany potrzebne konkretnie do modelu wykonam w transformacji.


In [13]:
df.dtypes


emp_title                               str
emp_length                          float64
state                                   str
homeownership                           str
annual_income                       float64
verified_income                         str
debt_to_income                      float64
annual_income_joint                 float64
verification_income_joint               str
debt_to_income_joint                float64
delinq_2y                             int64
months_since_last_delinq            float64
earliest_credit_line                  int64
inquiries_last_12m                    int64
total_credit_lines                    int64
open_credit_lines                     int64
total_credit_limit                    int64
total_credit_utilized                 int64
num_collections_last_12m              int64
num_historical_failed_to_pay          int64
months_since_90d_late               float64
current_accounts_delinq               int64
total_collection_amount_ever    

### Wnioski dotyczące typów danych

Po podstawowym czyszczeniu typy kolumn nie uległy istotnej zmianie.

Kolumny tekstowe mają typ `str`, a kolumny numeryczne są zapisane jako
`int64` lub `float64`.

Niektóre kolumny zawierające wartości całkowite mają typ `float64`,
ponieważ występują w nich brakujące wartości (`NaN`).

Na tym etapie nie zmieniam typów danych na siłę. Ewentualne konwersje
wykonam później, jeżeli będą potrzebne do dalszej analizy lub modelu.

## 9. Podsumowanie czyszczenia

Na tym etapie:
- wczytałem i obejrzałem dane,
- sprawdziłem target,
- policzyłem braki,
- usunąłem ewentualne wiersze bez targetu,
- sprawdziłem i usunąłem dokładne duplikaty,
- usunąłem zbędne spacje z `emp_title`,
- pozostałe braki świadomie zostawiłem do kolejnego etapu.

Teraz zapisuję wynik do osobnego pliku CSV.


In [14]:
OUTPUT_PATH.parent.mkdir(exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False)

print("Zapisano plik:", OUTPUT_PATH)
print("Końcowy kształt:", df.shape)


Zapisano plik: data/02_po_czyszczeniu.csv
Końcowy kształt: (10000, 55)
